In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon
    def forward(self, x):
        return x * torch.rsqrt(torch.mean(x**2, dim=1, keepdim=True) + self.epsilon)

In [ ]:
class EqualizedWeight(nn.Module):
    """Áp dụng Equalized Learning Rate bằng cách scale trọng số lúc chạy (runtime)"""
    def __init__(self, shape, gain=2):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(shape))
        self.scale = (gain / np.prod(shape[1:])) ** 0.5
    def forward(self):
        return self.weight * self.scale

In [ ]:
class EqualizedConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1, gain=2):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, kernel, kernel))
        self.bias = nn.Parameter(torch.zeros(out_ch))
        self.scale = (gain / (in_ch * kernel * kernel)) ** 0.5
        self.padding = padding

    def forward(self, x):
        return F.conv2d(x, self.weight * self.scale, self.bias, padding=self.padding)

In [ ]:
class AdaIN(nn.Module):
    def __init__(self, channels, w_dim=512):
        super().__init__()
        self.instance_norm = nn.InstanceNorm2d(channels)
        self.style_scale = nn.Linear(w_dim, channels)
        self.style_bias = nn.Linear(w_dim, channels)

    def forward(self, x, w):
        x = self.instance_norm(x)
        style_s = self.style_scale(w).unsqueeze(2).unsqueeze(3)
        style_b = self.style_bias(w).unsqueeze(2).unsqueeze(3)
        return style_s * x + style_b

In [ ]:
class MappingNetwork(nn.Module):
    def __init__(self, z_dim=512, w_dim=512):
        super().__init__()
        layers = [PixelNorm()]
        for _ in range(8):
            layers.extend([nn.Linear(z_dim, w_dim), nn.LeakyReLU(0.2)])
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

In [ ]:
class GenBlock(nn.Module):
    def __init__(self, in_ch, out_ch, w_dim=512):
        super().__init__()
        self.conv1 = EqualizedConv2d(in_ch, out_ch)
        self.adain1 = AdaIN(out_ch, w_dim)
        self.conv2 = EqualizedConv2d(out_ch, out_ch)
        self.adain2 = AdaIN(out_ch, w_dim)
        self.noise_weight1 = nn.Parameter(torch.zeros(1, out_ch, 1, 1))
        self.noise_weight2 = nn.Parameter(torch.zeros(1, out_ch, 1, 1))

    def forward(self, x, w, noise):
        # Layer 1
        x = self.conv1(x) + self.noise_weight1 * noise
        x = F.leaky_relu(self.adain1(x, w), 0.2)
        # Layer 2
        x = self.conv2(x) + self.noise_weight2 * noise
        x = F.leaky_relu(self.adain2(x, w), 0.2)
        return x

In [ ]:
class StyleGANGenerator(nn.Module):
    def __init__(self, z_dim=512, w_dim=512):
        super().__init__()
        self.mapping = MappingNetwork(z_dim, w_dim)
        # Bắt đầu với hằng số 4x4 (khác với GAN thông thường)
        self.starting_constant = nn.Parameter(torch.randn(1, 512, 4, 4))
        self.initial_adain = AdaIN(512, w_dim)
        
        # Danh sách các block cho từng độ phân giải (4->8, 8->16, ...)
        self.blocks = nn.ModuleList([
            GenBlock(512, 512), # 8x8
            GenBlock(512, 256), # 16x16
            GenBlock(256, 128)  # 32x32... (mở rộng tùy ý)
        ])
        self.to_rgb = nn.ModuleList([
            EqualizedConv2d(512, 3, kernel=1, padding=0), # 4x4
            EqualizedConv2d(512, 3, kernel=1, padding=0), # 8x8
            EqualizedConv2d(256, 3, kernel=1, padding=0)  # 16x16
        ])

    def forward(self, z, alpha, steps):
        """
        steps: giai đoạn hiện tại (0: 4x4, 1: 8x8, ...)
        alpha: hệ số fade-in (0->1) để chuyển tiếp mượt mà giữa các độ phân giải
        """
        w = self.mapping(z)
        x = self.initial_adain(self.starting_constant, w)
        
        if steps == 0:
            return self.to_rgb[0](x)

        for i in range(steps):
            upsampled = F.interpolate(x, scale_factor=2, mode='bilinear')
            x = self.blocks[i](upsampled, w, torch.randn_like(upsampled[:, :1, :, :]))

        # Cơ chế Progressive Growing Fade-in
        final_rgb = self.to_rgb[steps](x)
        if alpha < 1.0:
            prev_rgb = self.to_rgb[steps-1](upsampled)
            final_rgb = alpha * final_rgb + (1 - alpha) * prev_rgb
            
        return final_rgb

In [ ]:
def gradient_penalty(critic, real, fake, alpha, train_step, device):
    BATCH_SIZE, C, H, W = real.shape
    beta = torch.rand((BATCH_SIZE, 1, 1, 1)).repeat(1, C, H, W).to(device)
    interpolated_images = real * beta + fake.detach() * (1 - beta)
    interpolated_images.requires_grad_(True)

    # Tính điểm cho ảnh nội suy
    mixed_scores = critic(interpolated_images, alpha, train_step)

    gradient = torch.autograd.grad(
        inputs=interpolated_images,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradient = gradient.view(gradient.shape[0], -1)
    gradient_norm = gradient.norm(2, dim=1)
    gp = torch.mean((gradient_norm - 1) ** 2)
    return gp

In [ ]:
import os
import zipfile
import gdown
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

# --- CẤU HÌNH ---
DATA_URL = 'https://drive.google.com/uc?id=1O7m1010EJjLE5QxLZiM9Fpjs7Oj6e684'
ZIP_PATH = 'celeba.zip'
EXTRACT_PATH = 'celeba_data'
BATCH_SIZES = {4: 128, 8: 64, 16: 32, 32: 16, 64: 8} # Giảm batch size khi tăng res
CHANNELS_IMG = 3
Z_DIM = 512
W_DIM = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. TẢI VÀ GIẢI NÉN DỮ LIỆU ---
def prepare_data():
    if not os.path.exists(ZIP_PATH):
        print("Đang tải dữ liệu từ Google Drive...")
        gdown.download(DATA_URL, ZIP_PATH, quiet=False)
    
    if not os.path.exists(EXTRACT_PATH):
        print("Đang giải nén...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
        print("Hoàn tất!")

# --- 2. DATA LOADER BIẾN THIÊN (Dùng cho Progressive Growing) ---
def get_loader(image_size):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    # Lưu ý: ImageFolder yêu cầu dữ liệu nằm trong thư mục con (ví dụ: celeba_data/img/...)
    dataset = datasets.ImageFolder(root=EXTRACT_PATH, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZES.get(image_size, 8), shuffle=True)
    return loader

# --- 3. HÀM HUẤN LUYỆN (RÚT GỌN) ---
def train_step(gen, critic, loader, step, alpha, opt_gen, opt_critic, scaler_gen, scaler_critic):
    loop = tqdm(loader, leave=True)
    for batch_idx, (real, _) in enumerate(loop):
        real = real.to(DEVICE)
        cur_batch_size = real.shape[0]

        # Huấn luyện Critic (Discriminator)
        noise = torch.randn(cur_batch_size, Z_DIM).to(DEVICE)
        
        with torch.cuda.amp.autocast():
            fake = gen(noise, alpha, step)
            critic_real = critic(real, alpha, step)
            critic_fake = critic(fake.detach(), alpha, step)
            
            # WGAN-GP Loss (Đơn giản hóa)
            loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))

        opt_critic.zero_grad()
        scaler_critic.scale(loss_critic).backward()
        scaler_critic.step(opt_critic)
        scaler_critic.update()

        # Huấn luyện Generator
        with torch.cuda.amp.autocast():
            gen_fake = critic(fake, alpha, step)
            loss_gen = -torch.mean(gen_fake)

        opt_gen.zero_grad()
        scaler_gen.scale(loss_gen).backward()
        scaler_gen.step(opt_gen)
        scaler_gen.update()

        # Cập nhật Alpha (fade-in)
        # Alpha tăng dần từ 0 đến 1 trong suốt quá trình huấn luyện mỗi bước phân giải
        
        loop.set_description(f"Step [{step}]")
        loop.set_postfix(loss_c=loss_critic.item(), loss_g=loss_gen.item())

# --- 4. LUỒNG CHÍNH ---
def main():
    prepare_data()
    
    # Khởi tạo model (Giả sử bạn đã định nghĩa StyleGANGenerator và Discriminator từ bài trước)
    gen = StyleGANGenerator(Z_DIM, W_DIM).to(DEVICE)
    critic = StyleGANDiscriminator().to(DEVICE) # Bạn cần định nghĩa thêm class Discriminator tương ứng

    opt_gen = optim.Adam(gen.parameters(), lr=1e-3, betas=(0.0, 0.99))
    opt_critic = optim.Adam(critic.parameters(), lr=1e-3, betas=(0.0, 0.99))
    
    scaler_gen = torch.cuda.amp.GradScaler()
    scaler_critic = torch.cuda.amp.GradScaler()

    gen.train()
    critic.train()

    # Bắt đầu huấn luyện từ 4x4 lên 64x64
    step = 0 # 0: 4x4, 1: 8x8, 2: 16x16...
    for res in [4, 8, 16, 32, 64]:
        print(f"Đang huấn luyện độ phân giải: {res}x{res}")
        loader = get_loader(res)
        alpha = 1e-5 # Bắt đầu fade-in
        
        # Chạy một số epoch cho mỗi độ phân giải
        for epoch in range(10):
            train_step(gen, critic, loader, step, alpha, opt_gen, opt_critic, scaler_gen, scaler_critic)
            # Logic cập nhật alpha sau mỗi batch hoặc epoch ở đây
            alpha = min(alpha + 0.01, 1.0) 
            
        step += 1

if __name__ == "__main__":
    main()